In [ ]:
from pprint import pprint

In [ ]:
!pip install datasets==3.6.0

In [ ]:
import datasets

#checking dataset version
print(datasets.__version__)

#Q.No - 1

Choose all the correct statements about the dataset at the following URL

https://huggingface.co/datasets/ai4bharat/naamapadam


Note: We strongly recommend using the appropriate functions in the datasets module to obtain the dataset’s information, rather than accessing that information directly from the hub.

In [ ]:
from datasets import get_dataset_config_names, load_dataset

print("Total number of  subsets ",len(get_dataset_config_names("ai4bharat/naamapadam")))

hindi = load_dataset("ai4bharat/naamapadam","hi")

print("Total number of training samples", len(hindi['train']))

print("Total number of subclasses", len(hindi['train'].features['ner_tags'].feature.names))



#Q.No - 2
Common data

Download the Tamil sub-dataset from the following URL https://huggingface. co/datasets/ai4bharat/naamapadam, and store in a variable named “ds”.

Find the location of the cache directory (one way to determine the cache directory location is by using a method from the Dataset class; please refer to the documentation). Which of the following files are in the cached directory corresponding to the downloaded dataset?

In [ ]:
ds = load_dataset("ai4bharat/naamapadam","ta")

pprint(ds.cache_files)

#Q.No - 3
What is the disk space size of the downloaded dataset in MB?

In [ ]:
import os

size = 0

for file in ds.cache_files:
  for file_info in ds.cache_files[file]:
    size += os.path.getsize(file_info['filename'])

print("FIle size in MB",size/ (1024 * 1024))

#Q.No - 4
Enter the number of training samples in the dataset.

In [ ]:
print("Total number of  training samples",len(ds['train']))

#Q.No - 5
Create a new column named “num tokens” using the 'map' function.. Compute the number of tokens (words) in each sample and store the results in the newly created column. Reassign the modified dataset to the same variable “ds”. How many tokens (in millions) are there in the entire dataset?

In [ ]:
def num_tokens(rows):
  rows['num_tokens'] = len(rows['tokens'])
  return rows

In [ ]:
ds = ds.map(num_tokens)

In [ ]:
total = 0
def total_tokens(rows):
  global total
  total += rows['num_tokens']

ds.map(total_tokens)
print("Total tokens in million",toal/10_00_000)

#Q.No - 6
The statement that the modified dataset increased the disk space requirement about two fold is

In [ ]:
size = 0
for file in ds.cache_files:
  for name in ds.cache_files[file]:
    size += os.path.getsize(name['filename'])
print("Total new file size in MB", size/(1024*1024) )

In [ ]:
!ls -l ~/.cache/huggingface/datasets/ai4bharat___naamapadam/ta/1.0.0/9d4f21ac57d11ed4f9ea64854fdc9f5618e61acc

Note : Though the file size doesn't gets modified here, there exits a slight variation, one must understand that, every time when we use the map funtion, the dataset gets duplicated and the file size gets doubled every time. So the new file size is,

Size = 179 + 183 = 362 (almost double)

#Q.No - 7

Concatenate all the samples across the splits in the following order: [train:test:validation]. Currently, each sample contains a list of words (tokens). Create a sentence by joining the individual words (tokens) in each sample using a single white space as a delimiter and store the re sulting sample in a new column named “text”. Create a new dataset by removing the columns “ner tags” and “tokens”. Store the new dataset in the same variable “ds”. Enter the total number of samples in “ds”. Use map function.

In [ ]:
print("Total number of data sets in the subset is equal to ",(len(ds['train'])+len(ds['test'])+len(ds['validation'])))

In [ ]:
from datasets import concatenate_datasets

ds = concatenate_datasets([ds['train'],ds['test'],ds['validation']])

In [ ]:
def text(row):
  row['text'] = " ".join(row['tokens'])
  return row

In [ ]:
ds = ds.map(text)

In [ ]:
ds[0]

In [ ]:
ds = ds.remove_columns(["ner_tags","tokens"])

In [ ]:
ds[0]

#Q.No - 9
Filter the dataset so that all the samples in the dataset should have at least six tokens (any symbol separated by a white space is considered a token). Enter the number of samples in the dataset after filtering. Enter the exact number. If the answer is 123456, enter it as 123456.

In [ ]:
ds = ds.filter(lambda x: x['num_tokens']>=6)

In [ ]:
print("Dataset with atleast six tokens",len(ds))

#Q.No - 10
Download all the splits of Tamil sub-dataset “inltkh.ta” of https://huggingface.co/datasets/ai4bharat/indic_glue .

Filter the dataset so that each sample contains at least six words (separated by a single white space).Then Interleave the resultant dataset with the above filtered dataset of naamapadam. Take 80% of samples from naamapadam and 20% from indic glue. Enter the number of samples after interleaving the datasets.
Note: Set the value of the seed argument to 42.

In [ ]:
tamil_new = load_dataset("ai4bharat/indic_glue","inltkh.ta",split="train+test+validation")

In [ ]:
tamil_new

In [ ]:
tamil_new = tamil_new.filter(lambda x: len(x['text'].split(' '))>=6)

In [ ]:
tamil_new

In [ ]:
tamil_new = tamil_new.rename_column("label","num_tokens")

In [ ]:
tamil_new

In [ ]:
ds

In [ ]:
tamil_new = tamil_new.select_columns(["text"])
ds = ds.select_columns(["text"])

In [ ]:
tamil_new

In [ ]:
ds

In [ ]:
#Now we have to use the interleaving funtion to answer the above question.

from datasets import interleave_datasets

inter_data = interleave_datasets(
    datasets=[ds,tamil_new],
    probabilities=[0.8,0.2],
    seed=42
)

In [ ]:
print("Toal number of new samples after interleaving the dataset is equal to,",len(inter_data))

Additional Questions to ponder


Can we load a dataset directly from external links?

Can we load a dataset from zipped file directly?


In [ ]:
#To load dataset directly from external links

data = load_dataset("json", data_files="the corresponding link to the dataset")

#To load dataset from zip files

data = load_dataset("csv",data_files="the respective path to the zip file")